In [ ]:
!pip install -q -U langchain-google-genai pandas matplotlib gradio

In [ ]:
# ============================================================
# CELL 2 — IMPORTS + GEMINI
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from langchain_google_genai import ChatGoogleGenerativeAI

print("✅ Imports successful")


# ============================================================
# GEMINI API KEY
# ============================================================

API_KEY = "PASTE_YOUR_GEMINI_API_KEY_HERE"

if API_KEY == "AQ.Ab8RN6JopiYUtFBr2uhv7wSZ6JyKM46oe2LsXsvThoqCJETCvQ":
    raise ValueError(
        "❌ Please enter your Gemini API key."
    )

os.environ["GOOGLE_API_KEY"] = API_KEY


# ============================================================
# GEMINI MODEL
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    max_output_tokens=500
)

print("✅ Gemini configured")
print("⚠️ No Gemini request has been made yet.")

✅ Imports successful
✅ Gemini configured
⚠️ No Gemini request has been made yet.


In [ ]:
# ============================================================
# CELL 3 — DATA ANALYSIS FUNCTIONS
# ============================================================

df = None


# ============================================================
# LOAD CSV
# ============================================================

def load_csv(file):

    global df

    if file is None:
        return "❌ Please upload a CSV file."

    try:

        df = pd.read_csv(file)

        df = df.dropna(
            axis=1,
            how="all"
        )

        return f"""
# ✅ Dataset Loaded

### Dataset Information

**Rows:** {df.shape[0]}

**Columns:** {df.shape[1]}

### Columns

{chr(10).join("- " + str(c) for c in df.columns)}

### Preview

{df.head(5).to_markdown(index=False)}
"""

    except Exception as e:

        return f"""
# ❌ CSV Error

```text
{str(e)}
```
"""

In [ ]:
# ============================================================
# CELL 4 — COMPLETE GEMINI AI ANALYSIS
# ============================================================

print("🔄 Setting up Gemini AI...")


# ============================================================
# GEMINI QUESTION FUNCTION
# ============================================================

def ask_gemini(question):

    global df

    # --------------------------------------------------------
    # CHECK CSV
    # --------------------------------------------------------

    if df is None:
        return """
# ❌ No Dataset

Please upload your CSV file first and click:

**📂 Load Dataset**
"""


    # --------------------------------------------------------
    # CHECK QUESTION
    # --------------------------------------------------------

    if question is None or not str(question).strip():

        return """
# ❌ No Question

Please enter a question about your dataset.
"""


    try:

        # ====================================================
        # DATASET INFORMATION
        # ====================================================

        columns = ", ".join(
            [str(column) for column in df.columns]
        )


        # ====================================================
        # DATASET CONTENT
        # ====================================================
        # Your sample dataset is small, so we can send the
        # complete dataset to Gemini.
        # This avoids missing rows.
        # ====================================================

        if len(df) <= 30:

            dataset_text = df.to_csv(
                index=False
            )

        else:

            dataset_text = df.head(30).to_csv(
                index=False
            )


        # ====================================================
        # NUMERIC DATA
        # ====================================================

        numeric_columns = df.select_dtypes(
            include="number"
        )


        if len(numeric_columns.columns) > 0:

            numeric_summary = (
                numeric_columns
                .describe()
                .round(2)
                .to_string()
            )

        else:

            numeric_summary = "No numeric columns available."


        # ====================================================
        # DATASET SIZE
        # ====================================================

        rows = df.shape[0]

        columns_count = df.shape[1]


        # ====================================================
        # CREATE GEMINI PROMPT
        # ====================================================

        prompt = f"""
You are an expert AI Data Analyst.

Analyze the following CSV dataset and answer
the user's question accurately.

==================================================
DATASET INFORMATION
==================================================

Number of rows:
{rows}

Number of columns:
{columns_count}

Column names:
{columns}

==================================================
DATASET
==================================================

{dataset_text}

==================================================
NUMERIC STATISTICS
==================================================

{numeric_summary}

==================================================
USER QUESTION
==================================================

{question}

==================================================
INSTRUCTIONS
==================================================

1. Answer using only the supplied dataset.
2. Do not invent data.
3. Give exact numbers when possible.
4. Explain calculations briefly when useful.
5. Keep the answer easy to understand.
6. If the question cannot be answered from the dataset,
   clearly say that.
7. Act like a professional business data analyst.
"""


        # ====================================================
        # CALL GEMINI
        # ====================================================

        print("🤖 Sending one request to Gemini...")

        response = llm.invoke(
            prompt
        )


        # ====================================================
        # GET RESPONSE
        # ====================================================

        answer = response.content


        # ====================================================
        # RETURN ANSWER
        # ====================================================

        return f"""
# 🤖 Gemini Analysis

{answer}
"""


    # ========================================================
    # ERROR HANDLING
    # ========================================================

    except Exception as e:

        error_message = str(e)


        # ----------------------------------------------------
        # 429 QUOTA ERROR
        # ----------------------------------------------------

        if (
            "429" in error_message
            or "RESOURCE_EXHAUSTED" in error_message
            or "quota" in error_message.lower()
        ):

            return """
# ⚠️ Gemini API Quota Reached

Your Gemini API project has reached its
current request quota.

### You can still use:

✅ CSV upload
✅ Dataset summary
✅ Bar chart
✅ Line chart
✅ Histogram
✅ Scatter plot

Only the Gemini AI question feature is affected.

Please wait for your Gemini quota to reset
or use a project with available quota.
"""


        # ----------------------------------------------------
        # 503 ERROR
        # ----------------------------------------------------

        elif (
            "503" in error_message
            or "UNAVAILABLE" in error_message
        ):

            return """
# ⚠️ Gemini Temporarily Unavailable

Gemini is currently experiencing high demand.

Please wait a little and try again.
"""


        # ----------------------------------------------------
        # 404 MODEL ERROR
        # ----------------------------------------------------

        elif (
            "404" in error_message
            or "NOT_FOUND" in error_message
        ):

            return f"""
# ❌ Gemini Model Error

The Gemini model could not be found or is
not available for your API project.

### Error

```text
{error_message}
```
"""

🔄 Setting up Gemini AI...


In [ ]:
# ============================================================
# CELL 5 — GEMINI AI DATA ANALYST WEBSITE
# ============================================================

import gradio as gr

print("🚀 Creating AI Data Analyst website...")


# ============================================================
# DATASET SUMMARY FUNCTION
# ============================================================

def dataset_summary():

    global df

    if df is None:
        return """
# ❌ No Dataset

Please upload your CSV file first and click:

**📂 Load Dataset**
"""

    # Implement actual summary generation here later
    return f"""
# ✅ Dataset Summary

This is a placeholder for the dataset summary.

**Rows:** {df.shape[0]}
**Columns:** {df.shape[1]}
"""


# ============================================================
# CHART GENERATOR FUNCTION
# ============================================================

def generate_chart(chart_type):

    global df

    if df is None:
        return """
# ❌ No Dataset

Please upload your CSV file first and click:

**📂 Load Dataset**
"""

    # Implement chart generation logic here later
    return f"""
# 📈 Chart Generation Placeholder

**Chart Type Requested:** {chart_type}

Chart generation logic will be implemented here.
"""


# ============================================================
# CREATE GRADIO APP
# ============================================================

with gr.Blocks(
    title="Gemini AI Data Analyst"
) as app:

    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown("""
# 📊 Gemini AI Data Analyst

### Upload CSV → Analyze Dataset → Generate Charts → Ask Gemini

This application uses:

- 🐍 Python
- 📊 Pandas
- 📈 Matplotlib
- 🤖 Google Gemini
- 🔗 LangChain
- 🌐 Gradio
""")


    # ========================================================
    # 1. CSV UPLOAD
    # ========================================================

    gr.Markdown("---")
    gr.Markdown("## 📁 1. Upload CSV Dataset")

    with gr.Row():

        file_input = gr.File(
            label="Choose your CSV file",
            file_types=[".csv"],
            type="filepath"
        )

        load_button = gr.Button(
            "📂 Load Dataset",
            variant="primary"
        )


    dataset_info = gr.Markdown(
        """
### 📂 No dataset loaded

Upload a CSV file and click **Load Dataset**.
"""
    )


    load_button.click(
        fn=load_csv,
        inputs=file_input,
        outputs=dataset_info
    )


    # ========================================================
    # 2. DATASET SUMMARY
    # ========================================================

    gr.Markdown("---")
    gr.Markdown("## 📊 2. Dataset Summary")

    summary_button = gr.Button(
        "📊 Generate Summary",
        variant="primary"
    )


    summary_output = gr.Markdown(
        "Summary will appear here."
    )


    summary_button.click(
        fn=dataset_summary,
        inputs=[],
        outputs=summary_output
    )


    # ========================================================
    # 3. CHART GENERATOR
    # ========================================================

    gr.Markdown("---")
    gr.Markdown("## 📈 3. Generate Charts")


    chart_type = gr.Dropdown(
        choices=[
            "Bar Chart",
            "Line Chart",
            "Histogram",
            "Scatter Plot"
        ],
        value="Bar Chart",
        label="Select Chart Type"
    )


    chart_button = gr.Button(
        "📈 Generate Chart",
        variant="primary"
    )


    chart_output = gr.Image(
        label="Generated Chart"
    )


    chart_button.click(
        fn=generate_chart,
        inputs=chart_type,
        outputs=chart_output
    )


    # ========================================================
    # 4. GEMINI AI
    # ========================================================

    gr.Markdown("---")
    gr.Markdown("## 🤖 4. Ask Gemini")


    gr.Markdown("""
### 💡 Example Questions

Try questions like:

**1.** Which product has the highest sales?

**2.** Which product has the highest profit?

**3.** Which region has the highest sales?

**4.** What are the main patterns in this dataset?

**5.** Give me a business analysis of this dataset?

**6.** Explain this dataset in simple words?

**7.** What product should the business focus on based on sales?
""")


    question = gr.Textbox(
        label="Ask a question",
        placeholder="Example: Which product has the highest sales?",
        lines=3
    )


    ask_button = gr.Button(
        "🤖 Ask Gemini",
        variant="primary"
    )


    gemini_output = gr.Markdown(
        """
### 🤖 Gemini Answer

Your answer will appear here.
"""
    )


    ask_button.click(
        fn=ask_gemini,
        inputs=question,
        outputs=gemini_output
    )


    # ========================================================
    # FOOTER
    # ========================================================

    gr.Markdown("""
---

## 🛠️ AI Data Analyst

**Pandas** → Data processing
**Matplotlib** → Charts
**Gemini** → AI analysis
**LangChain** → Gemini integration
**Gradio** → Web interface

### ⚡ Gemini API Usage

Gemini is called **only when you click "Ask Gemini"**.

CSV loading, summaries, and charts run locally.
""")


# ============================================================
# LAUNCH
# ============================================================

print("✅ Website created")
print("🚀 Starting Gradio...")
print("")


app.launch(
    share=True,
    debug=True,
    theme=gr.themes.Soft()
)

🚀 Creating AI Data Analyst website...


/tmp/ipykernel_1216/1553090211.py:69: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Website created
🚀 Starting Gradio...

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://599f27ccfe7445f6a2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
